# Feature Geometry

# Feature Geometry

## What This Is
The **linear representation hypothesis** (LRH) proposes that neural network features are linearly encoded in activation space — that is, there exist directions in activation space such that projecting onto that direction gives the intensity of the feature.

Evidence:
- Word2Vec analogies (king - man + woman = queen) work via linear arithmetic
- Probing classifiers: linear classifiers can decode many semantic features from internal activations
- Steering vectors work (NB 09)

## Privileged Basis
Most neural networks don't have a *privileged basis* — the coordinate axes of activation space don't correspond to interpretable features. Features are encoded in arbitrary directions. This is in contrast to some architectures (activation functions, LayerNorm) that do have a privileged basis.

In [ ]:
import numpy as np
from typing import List, Tuple

np.random.seed(42)

# Feature geometry analysis via PCA and ICA

D_MODEL = 64
N_SAMPLES = 500

# Generate activations with known latent structure
N_FEATURES = 10
# True features live in arbitrary directions
feature_directions = np.linalg.qr(np.random.randn(D_MODEL, N_FEATURES))[0]  # orthonormal
# Sparse feature activations
feature_activations = np.random.exponential(1, (N_SAMPLES, N_FEATURES)) * (np.random.random((N_SAMPLES, N_FEATURES)) > 0.7)
# Mix features into activation space
X_acts = feature_activations @ feature_directions.T + np.random.randn(N_SAMPLES, D_MODEL) * 0.1

# PCA: finds directions of maximum variance
X_c = X_acts - X_acts.mean(0)
cov = X_c.T @ X_c / len(X_c)
vals, vecs = np.linalg.eigh(cov)
top_pcs = vecs[:, np.argsort(vals)[-N_FEATURES:]]

# How well do PCs align with true features?
pc_feature_align = np.abs(top_pcs.T @ feature_directions)
max_pc_align = pc_feature_align.max(axis=1).mean()

print('Feature Geometry Analysis')
print('=' * 45)
print(f'True features: {N_FEATURES} in {D_MODEL}D space')
print(f'PCA top-{N_FEATURES} components avg alignment with true features: {max_pc_align:.3f}')
print('(1.0 = perfect recovery, ~0 = features not aligned with PCs)')

# Probing: can a linear probe decode feature activations from X?
def linear_probe_accuracy(X, y):
    w = np.linalg.lstsq(X, y, rcond=None)[0]
    pred = (X @ w > 0.5).astype(int)
    true_bin = (y > 0.5).astype(int)
    return (pred == true_bin).mean()

print('\nLinear Probe Accuracy (can we decode features from activations?):')
for i in range(min(4, N_FEATURES)):
    acc = linear_probe_accuracy(X_acts, feature_activations[:, i])
    print(f'  Feature {i}: probe accuracy = {acc:.3f}')

print('\nKey insight: linear probes work even when PCA doesnt perfectly align,')
print('confirming the linear representation hypothesis.')
